# SAM-3D Body Pipeline

Runs pose estimation on an image and saves outputs to `data/output/`.

**First time?** Run cells 1–3 to install everything. After that you only need cells 4+.

## ① Setup

In [ ]:
# Can skip if not using your Google Drive 

# from google.colab import drive, files
# drive.mount('/content/drive')


In [ ]:
# Clone SAM-3D Body and SAM3 (Meta's upstream repos)
!git clone https://github.com/facebookresearch/sam-3d-body.git
!git clone https://github.com/facebookresearch/sam3.git

In [ ]:
# Install SAM3 in editable mode (must be done before other deps)
%cd sam3
!pip install -e . --quiet
%cd ..

In [ ]:
# Install all project dependencies
!pip install -r sam3dbody_pipeline/requirements.txt --quiet

# SAM-3D Body specific deps
!pip install pytorch-lightning pyrender yacs scikit-image einops timm hydra-core roma --quiet
!pip install 'git+https://github.com/facebookresearch/detectron2.git@a1ce2f9' --no-build-isolation --no-deps --quiet
!pip install git+https://github.com/microsoft/MoGe.git --quiet

# Pin numpy (required by SAM3D)
!pip install numpy==1.26.4 --force-reinstall --quiet

print('All dependencies installed. Restart kernel before continuing.')

## ② Load Model

In [ ]:
!ls sam3dbody_pipeline  

In [ ]:
import sys
sys.path.insert(0, '..')

from sam3db.pipeline import load_model

# Reads HF_TOKEN from .env automatically
estimator, visualizer = load_model()

## ③ Run Inference

In [ ]:
import cv2
import matplotlib.pyplot as plt
from sam3db.pipeline import run_inference

IMAGE_PATH = '../data/input/your_image.png'  # ← change this

img_cv2 = cv2.imread(IMAGE_PATH)
img_rgb = cv2.cvtColor(img_cv2, cv2.COLOR_BGR2RGB)

outputs = run_inference(estimator, IMAGE_PATH)

plt.figure(figsize=(8, 5))
plt.imshow(img_rgb)
plt.axis('off')
plt.title('Input Image')
plt.show()

## ④ Save Outputs

In [ ]:
import pickle
import os
from pathlib import Path

image_name = Path(IMAGE_PATH).stem
output_path = f'../data/output/{image_name}_outputs.pkl'

os.makedirs('../data/output', exist_ok=True)
with open(output_path, 'wb') as f:
    pickle.dump(outputs, f)

print(f'Saved to {output_path}')

## ⑤ 2D Keypoint Visualization

In [ ]:
# Uses the upstream utils (from sam-3d-body/notebook/)
import sys
sys.path.insert(0, '../sam-3d-body/notebook')
from utils import visualize_2d_results, display_results_grid

if outputs:
    vis_results = visualize_2d_results(img_cv2, outputs, visualizer)
    titles = [f'Person {i} — 2D Keypoints' for i in range(len(vis_results))]
    display_results_grid(vis_results, titles, figsize_per_image=(6, 6))
else:
    print('No people detected.')